In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-darkgrid')

# Assets were selected across uncorrelated classes — equities, bonds,
# commodities, and real estate — so the efficient frontier reflects genuine
# diversification benefits rather than variance within a single sector.
tickers = ['SPY', 'AAPL', 'GLD', 'TLT', 'VNQ', 'EFA', 'XLE']
start_date = '2020-01-01'

print(f"Downloading data for: {tickers}")
data = yf.download(tickers, start=start_date, progress=False, auto_adjust=True)

if isinstance(data.columns, pd.MultiIndex):
    try:
        prices = data['Close']
    except KeyError:
        prices = data

prices = prices.dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

normalized_prices = (prices / prices.iloc[0]) * 100

plt.figure(figsize=(12, 6))
plt.plot(normalized_prices)
plt.title("Portfolio Asset Performance (Normalized to 100)")
plt.xlabel("Date")
plt.ylabel("Normalized Price")
plt.legend(normalized_prices.columns)
plt.show()

print("Data fetched successfully. Current dimensions:", log_returns.shape)

In [ ]:
# 1. Calculate Annualized Metrics
# 252 trading days in a year
mean_returns = log_returns.mean() * 252
cov_matrix = log_returns.cov() * 252

print("--- Annualized Expected Returns ---")
print(mean_returns.apply(lambda x: f"{x:.2%}"))

# 2. Simulation Settings
num_portfolios = 10000
num_assets = len(tickers)

# Arrays to store the results
results = np.zeros((3, num_portfolios))
# Array to store the weights for each portfolio
all_weights = np.zeros((num_portfolios, num_assets))

np.random.seed(42) # For reproducibility

print("\nSimulating 10,000 portfolios...")

for i in range(num_portfolios):
    # a. Generate Random Weights
    weights = np.random.random(num_assets)
    weights /= np.sum(weights) # Normalize so they sum to 1 (100%)
    
    all_weights[i, :] = weights
    
    # b. Calculate Portfolio Return
    # Formula: w * mu
    p_return = np.sum(mean_returns * weights)
    
    # c. Calculate Portfolio Volatility (Standard Deviation)
    # Formula: sqrt( w.T * Cov * w )
    p_std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    # d. Calculate Sharpe Ratio (assuming Risk-Free Rate = 0 for simplicity)
    # Formula: (Rp - Rf) / Sigma_p
    results[0,i] = p_return
    results[1,i] = p_std_dev
    results[2,i] = results[0,i] / results[1,i] # Sharpe Ratio

print("Simulation Complete.")

In [ ]:
# 1. Find the index of the Portfolio with the Max Sharpe Ratio
max_sharpe_idx = np.argmax(results[2])

# Extract the data for that portfolio
max_sharpe_return = results[0, max_sharpe_idx]
max_sharpe_vol = results[1, max_sharpe_idx]
max_sharpe_weights = all_weights[max_sharpe_idx, :]

# 2. Print the Optimal Weights
print("--- Optimal Portfolio (Max Sharpe Ratio) ---")
print(f"Return: {max_sharpe_return:.2%}")
print(f"Volatility: {max_sharpe_vol:.2%}")
print(f"Sharpe Ratio: {results[2, max_sharpe_idx]:.2f}")
print("\nAllocation:")
for ticker, weight in zip(tickers, max_sharpe_weights):
    print(f"{ticker}: {weight:.2%}")

# 3. Visualizing the Efficient Frontier
plt.figure(figsize=(10, 6))

# Scatter plot of all simulated portfolios
# X-axis = Risk (Volatility), Y-axis = Return
plt.scatter(results[1,:], results[0,:], c=results[2,:], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')

# Highlight the Optimal Portfolio (Red Star)
plt.scatter(max_sharpe_vol, max_sharpe_return, c='red', s=200, marker='*', label='Max Sharpe')

plt.title('Efficient Frontier: 10,000 Random Portfolios')
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Return')
plt.legend()
plt.show()

In [ ]:
# 1. Find the index of the Portfolio with the Minimum Volatility
min_vol_idx = np.argmin(results[1])

# Extract data
min_vol_return = results[0, min_vol_idx]
min_vol_vol = results[1, min_vol_idx]
min_vol_weights = all_weights[min_vol_idx, :]

# 2. Print the Safe Portfolio
print("--- Global Minimum Variance (GMV) Portfolio ---")
print(f"Return: {min_vol_return:.2%}")
print(f"Volatility: {min_vol_vol:.2%}") # This should be lower than the Max Sharpe Volatility
print("\nAllocation:")
for ticker, weight in zip(tickers, min_vol_weights):
    print(f"{ticker}: {weight:.2%}")

# 3. Updated Plot with BOTH Stars
plt.figure(figsize=(10, 6))

# Scatter plot
plt.scatter(results[1,:], results[0,:], c=results[2,:], cmap='viridis', alpha=0.5, s=10)
plt.colorbar(label='Sharpe Ratio')

# Max Sharpe (Red Star)
plt.scatter(max_sharpe_vol, max_sharpe_return, c='red', s=200, marker='*', label='Max Sharpe (Aggressive)')

# Min Volatility (Blue Star)
plt.scatter(min_vol_vol, min_vol_return, c='blue', s=200, marker='*', label='Min Vol (Conservative)')

plt.title('Efficient Frontier: Risk vs. Reward')
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Return')
plt.legend()
plt.show()